# 🔎 Basic RAG System (with Reranker) — Multi-LLM-Provider Edition

This notebook implements a complete Retrieval-Augmented Generation (RAG) pipeline, following these steps:

1. Collect documents (PDF, Word, txt, websites, APIs)
2. Clean and prepare text
3. Chunk text (with overlap, context preserved)
4. Embed chunks (sentence-transformers, normalized)
5. Store embeddings in a vector database (FAISS)
6. Receive the user's question (optional query rewriting)
7. Embed the question
8. Retrieve top-K chunks (similarity + optional hybrid BM25 search)
9. Rerank retrieved chunks (Cross-Encoder)
10. Build the LLM context (dedupe, format with sources)
11. Send the context to the LLM (strict system prompt)
12. Generate the final answer (with citations, and "not found" handling)

**LLM Provider Switch:** Change one config cell to swap between **Ollama** (local, free), **OpenRouter**, **Groq**, **OpenAI**, or any other OpenAI-compatible API. No other code needs to change.

> ⚠️ Run cells top to bottom. Install dependencies first (Step 0).


## Step 0 — Install dependencies

In [ ]:
%pip install -q sentence-transformers faiss-cpu rank_bm25 pymupdf python-docx \
    beautifulsoup4 requests numpy tqdm ipykernel
print("Dependencies installed.")

## Step 0.1 — Configuration: pick your LLM provider

Supported out of the box (all speak the OpenAI-style `/chat/completions` schema, so switching is just config):

| Provider    | `base_url`                                         | Needs API key? | Example model name                  |
|-------------|-----------------------------------------------------|----------------|--------------------------------------|
| `ollama`    | `http://localhost:11434/v1`                         | No             | `llama3.1`, `qwen2.5`, `mistral`     |
| `openrouter`| `https://openrouter.ai/api/v1`                       | Yes            | `openai/gpt-4o-mini`, `anthropic/claude-3.5-sonnet`, `meta-llama/llama-3.1-70b-instruct` |
| `groq`      | `https://api.groq.com/openai/v1`                     | Yes            | `llama-3.3-70b-versatile`, `mixtral-8x7b-32768` |
| `openai`    | `https://api.openai.com/v1`                          | Yes            | `gpt-4o-mini`, `gpt-4o`              |
| `custom`    | any OpenAI-compatible endpoint you run yourself      | maybe          | whatever your server exposes         |

Just edit `LLM_PROVIDER` and `LLM_MODEL` below. Put API keys in environment variables instead of hardcoding them when possible.


In [ ]:
import os
import getpass

# ---- 1) Choose your provider: "ollama" | "openrouter" | "groq" | "openai" | "custom"
LLM_PROVIDER = "ollama"

# ---- 2) Choose the model name for that provider (see table above)
LLM_MODEL = "llama3.2"

# ---- 3) Provider registry (edit / extend freely) ----
PROVIDERS = {
    "ollama": {
        "base_url": "http://localhost:11434/v1",
        "needs_key": False,
        "env_var": None,
    },
    "openrouter": {
        "base_url": "https://openrouter.ai/api/v1",
        "needs_key": True,
        "env_var": "OPENROUTER_API_KEY",
    },
    "groq": {
        "base_url": "https://api.groq.com/openai/v1",
        "needs_key": True,
        "env_var": "GROQ_API_KEY",
    },
    "openai": {
        "base_url": "https://api.openai.com/v1",
        "needs_key": True,
        "env_var": "OPENAI_API_KEY",
    },
    "custom": {
        "base_url": "http://localhost:8000/v1",   # <-- point this at your own server
        "needs_key": False,
        "env_var": None,
    },
}

cfg = PROVIDERS[LLM_PROVIDER]
api_key = None
if cfg["needs_key"]:
    api_key = os.environ.get(cfg["env_var"])
    if not api_key:
        api_key = getpass.getpass(f"Enter your {LLM_PROVIDER} API key ({cfg['env_var']}): ")
        os.environ[cfg["env_var"]] = api_key

print(f"Provider: {LLM_PROVIDER} | Model: {LLM_MODEL} | Base URL: {cfg['base_url']}")


## Unified LLM client

One function, `chat()`, talks to *any* of the providers above because they all implement the OpenAI
`/chat/completions` schema (Ollama supports this natively since 0.1.x under `/v1`). Only the base URL
and the auth header change.


In [ ]:
import requests

def chat(messages, temperature=0.2, max_tokens=1000, provider=None, model=None):
    """
    Send a chat completion request to whichever provider is configured.
    messages: list of {"role": "system"|"user"|"assistant", "content": str}
    """
    provider = provider or LLM_PROVIDER
    model = model or LLM_MODEL
    cfg = PROVIDERS[provider]

    url = f"{cfg['base_url'].rstrip('/')}/chat/completions"
    headers = {"Content-Type": "application/json"}

    if cfg["needs_key"]:
        key = os.environ.get(cfg["env_var"])
        headers["Authorization"] = f"Bearer {key}"
    elif provider == "ollama":
        # Ollama ignores the header but some proxies want something present
        headers["Authorization"] = "Bearer ollama"

    # OpenRouter likes to know who is calling (optional but good practice)
    if provider == "openrouter":
        headers["HTTP-Referer"] = "https://localhost"
        headers["X-Title"] = "Basic RAG Notebook"

    payload = {
        "model": model,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
    }

    resp = requests.post(url, headers=headers, json=payload, timeout=120)
    if resp.status_code != 200:
        raise RuntimeError(f"LLM call failed [{resp.status_code}]: {resp.text[:500]}")

    data = resp.json()
    return data["choices"][0]["message"]["content"]


# Quick smoke test
try:
    test_reply = chat([{"role": "user", "content": "Reply with exactly: RAG notebook ready."}], max_tokens=20)
    print("LLM connection OK ->", test_reply)
except Exception as e:
    print("LLM connection FAILED. Check your provider/model/API key/local server.")
    print(e)


## Step 1 — Collect the documents

Loaders for the common source types: PDF, Word (.docx), plain text/markdown, a website URL, or a
generic JSON API response. Each loader returns a list of `{"text": ..., "source": ...}` dicts so we
can keep provenance for citations later.


In [12]:
from pathlib import Path
import pymupdf
import docx
import csv
import json

def load_pdf(path):
    """Load PDF files page by page using PyMuPDF."""
    docs = []
    with pymupdf.open(path) as doc:
        for i, page in enumerate(doc):
            text = page.get_text() or ""
            if text.strip():
                docs.append({"text": text, "source": f"{Path(path).name} (page {i+1})"})
    return docs

def load_docx(path):
    """Load Word (.docx) files."""
    d = docx.Document(path)
    full_text = "\n".join(p.text for p in d.paragraphs if p.text.strip())
    return [{"text": full_text, "source": Path(path).name}]

def load_txt(path):
    """Load plain text or markdown files."""
    text = Path(path).read_text(encoding="utf-8", errors="ignore")
    return [{"text": text, "source": Path(path).name}]

def load_csv(path):
    """Load CSV files and treat each row as a document."""
    docs = []
    with open(path, encoding="utf-8", errors="ignore") as f:
        reader = csv.DictReader(f)
        for i, row in enumerate(reader):
            row_text = " | ".join(f"{k}: {v}" for k, v in row.items() if v)
            if row_text.strip():
                docs.append({"text": row_text, "source": f"{Path(path).name} (row {i+1})"})
    return docs

def load_json(path):
    """Load JSON files and convert to formatted text."""
    with open(path, encoding="utf-8", errors="ignore") as f:
        data = json.load(f)
    text = json.dumps(data, indent=2, ensure_ascii=False) if isinstance(data, (dict, list)) else str(data)
    return [{"text": text, "source": Path(path).name}]

def load_folder(folder_path="D:/rag/nuclear-law-rag/documents"):
    """
    Scan data directory, auto-detect file formats, and load all supported documents.
    """
    folder = Path(folder_path)
    if not folder.exists():
        print(f"⚠️ Directory not found: {folder_path}")
        return []

    docs = []
    all_files = [p for p in folder.glob("**/*") if p.is_file()]
    print(f"📂 Found {len(all_files)} file(s) in: '{folder_path}'\n" + "-" * 40)

    for p in all_files:
        ext = p.suffix.lower()
        try:
            loaded = []
            if ext == ".pdf":
                loaded = load_pdf(str(p))
            elif ext in (".docx", ".doc"):
                loaded = load_docx(str(p))
            elif ext in (".txt", ".md", ".log"):
                loaded = load_txt(str(p))
            elif ext == ".csv":
                loaded = load_csv(str(p))
            elif ext == ".json":
                loaded = load_json(str(p))
            else:
                print(f"⏩ Skipping unsupported file: {p.name} ({ext})")
                continue

            docs.extend(loaded)
            print(f"✅ Loaded: {p.name} [{ext}] -> ({len(loaded)} item(s)/page(s))")

        except Exception as e:
            print(f"❌ Error loading {p.name}: {e}")

    print("-" * 40)
    return docs

print("Loaders ready: PDF, DOCX, TXT, CSV, JSON, and load_folder")

Loaders ready: PDF, DOCX, TXT, CSV, JSON, and load_folder


In [13]:
DATA_DIR = Path("documents") if Path("documents").exists() else Path("data")

RAW_DOCS = load_folder("D:/rag/nuclear-law-rag/documents")

if not RAW_DOCS:
    print("⚠️ No valid documents found in the data directory!")
else:
    print(f"\n🎉 Total documents loaded successfully: {len(RAW_DOCS)}")

📂 Found 1 file(s) in: 'D:/rag/nuclear-law-rag/documents'
----------------------------------------
✅ Loaded: Handbook on Nuclear Law.pdf [.pdf] -> (174 item(s)/page(s))
----------------------------------------

🎉 Total documents loaded successfully: 174


## Step 2 — Clean and prepare the text

Strip boilerplate headers/footers, repeated whitespace, page-number artifacts, and de-duplicate
near-identical documents.


In [52]:
import re
from collections import Counter, defaultdict

def normalize_line(line):
    return re.sub(r"\s+", " ", line.strip().lower())

def get_edge_lines(doc, edge_lines=3):
    lines = [line.strip() for line in doc["text"].splitlines() if line.strip()]

    if not lines:
        return [], []

    return lines[:edge_lines], lines[-edge_lines:]


def detect_repeated_edges(docs, edge_lines=3, min_repeat_count=4):
    top_data = defaultdict(list)
    bottom_data = defaultdict(list)

    for doc_index, doc in enumerate(docs):
        top, bottom = get_edge_lines(doc, edge_lines)

        for line in top:
            normalized = normalize_line(line)

            if len(normalized) >= 6:
                top_data[normalized].append({
                    "doc_index": doc_index,
                    "source": doc.get("source", ""),
                    "text": line
                })

        for line in bottom:
            normalized = normalize_line(line)

            if len(normalized) >= 6:
                bottom_data[normalized].append({
                    "doc_index": doc_index,
                    "source": doc.get("source", ""),
                    "text": line
                })

    repeated_top = {
        line: items
        for line, items in top_data.items()
        if len(items) >= min_repeat_count
    }

    repeated_bottom = {
        line: items
        for line, items in bottom_data.items()
        if len(items) >= min_repeat_count
    }

    return repeated_top, repeated_bottom

In [53]:
REPEATED_TOP, REPEATED_BOTTOM = detect_repeated_edges(
    RAW_DOCS,
    edge_lines=3,
    min_repeat_count=4
)

print("=" * 70)
print("REPEATED TOP/BOTTOM CANDIDATES")
print("=" * 70)

print("\nTOP CANDIDATES")
print("-" * 70)

for line, items in sorted(
    REPEATED_TOP.items(),
    key=lambda x: len(x[1]),
    reverse=True
):
    print(f"\n[{len(items)}x] {items[0]['text']}")
    
    pages = [
        item["source"]
        for item in items[:10]
    ]
    
    for page in pages:
        print(f"   → {page}")

    if len(items) > 10:
        print(f"   ... and {len(items) - 10} more")


print("\n" + "=" * 70)
print("BOTTOM CANDIDATES")
print("=" * 70)

for line, items in sorted(
    REPEATED_BOTTOM.items(),
    key=lambda x: len(x[1]),
    reverse=True
):
    print(f"\n[{len(items)}x] {items[0]['text']}")
    
    pages = [
        item["source"]
        for item in items[:10]
    ]
    
    for page in pages:
        print(f"   → {page}")

    if len(items) > 10:
        print(f"   ... and {len(items) - 10} more")

REPEATED TOP/BOTTOM CANDIDATES

TOP CANDIDATES
----------------------------------------------------------------------

BOTTOM CANDIDATES

[25x] PART III. NUCLEAR AND RADIATION SAFETY
   → Handbook on Nuclear Law.pdf (page 69)
   → Handbook on Nuclear Law.pdf (page 70)
   → Handbook on Nuclear Law.pdf (page 71)
   → Handbook on Nuclear Law.pdf (page 73)
   → Handbook on Nuclear Law.pdf (page 75)
   → Handbook on Nuclear Law.pdf (page 76)
   → Handbook on Nuclear Law.pdf (page 77)
   → Handbook on Nuclear Law.pdf (page 79)
   → Handbook on Nuclear Law.pdf (page 81)
   → Handbook on Nuclear Law.pdf (page 83)
   ... and 15 more

[24x] PART I. ELEMENTS OF NUCLEAR LAW
   → Handbook on Nuclear Law.pdf (page 21)
   → Handbook on Nuclear Law.pdf (page 23)
   → Handbook on Nuclear Law.pdf (page 25)
   → Handbook on Nuclear Law.pdf (page 27)
   → Handbook on Nuclear Law.pdf (page 29)
   → Handbook on Nuclear Law.pdf (page 31)
   → Handbook on Nuclear Law.pdf (page 33)
   → Handbook on Nuclear Law

In [54]:
def remove_detected_edges(
    docs,
    repeated_top,
    repeated_bottom,
    edge_lines=3
):
    cleaned_docs = []

    for doc in docs:
        lines = doc["text"].splitlines()

        non_empty = [
            i for i, line in enumerate(lines)
            if line.strip()
        ]

        if not non_empty:
            continue

        top_indexes = set(non_empty[:edge_lines])
        bottom_indexes = set(non_empty[-edge_lines:])

        new_lines = []

        for i, line in enumerate(lines):
            normalized = normalize_line(line)

            if i in top_indexes and normalized in repeated_top:
                continue

            if i in bottom_indexes and normalized in repeated_bottom:
                continue

            new_lines.append(line)

        cleaned_docs.append({
            **doc,
            "text": "\n".join(new_lines)
        })

    return cleaned_docs


DOCS_NO_HEADERS = remove_detected_edges(
    RAW_DOCS,
    REPEATED_TOP,
    REPEATED_BOTTOM,
    edge_lines=3
)

print(
    f"Headers/footers removed from "
    f"{len(DOCS_NO_HEADERS)} document(s)."
)

Headers/footers removed from 174 document(s).


In [59]:
import re
import hashlib

def clean_text(text: str) -> str:

    # normalize line endings
    text = re.sub(r"\r\n?", "\n", text)

    # normalize quotes and dashes
    text = (
        text.replace("“", '"')
            .replace("”", '"')
            .replace("‘", "'")
            .replace("’", "'")
            .replace("—", "-")
            .replace("–", "-")
            .replace("−", "-")
    )

    # normalize bullets
    text = re.sub(r"[\u2022\u2023\u25E6\u2043\u2219]", "- ", text)

    # remove page numbers
    text = re.sub(r"^\s*\d+\s*$", "", text, flags=re.M)
    text = re.sub(r"^\s*Page\s+\d+(?:\s+of\s+\d+)?\s*$", "", text, flags=re.I | re.M)
    text = re.sub(r"^\s*[ivxlcdm]+\s*$", "", text, flags=re.I | re.M)

    # fix broken words
    def fix_hyphen(match):
        prefix, suffix = match.group(1), match.group(2)

        if prefix.lower() in {
            "non", "self", "co", "cross",
            "multi", "pre", "sub", "re"
        }:
            return f"{prefix}-{suffix}"

        return f"{prefix}{suffix}"

    text = re.sub(
        r"(\b[a-zA-Z]+)-\n\s*([a-zA-Z]+\b)",
        fix_hyphen,
        text
    )

    # join broken sentence lines
    text = re.sub(
        r"(?<![.!?:])\n\s*(?=['\"a-z])",
        " ",
        text
    )

    # fix punctuation spacing
    text = re.sub(r"([.!?])(?=[A-Z])", r"\1 ", text)

    # normalize spaces
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n[ \t]+", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


def text_hash(text: str) -> str:
    normalized = re.sub(r"\s+", " ", text.strip().lower())
    return hashlib.md5(normalized.encode("utf-8")).hexdigest()


def dedupe_docs(docs):

    seen = set()
    unique = []

    for d in docs:

        text = d["text"].strip()

        if not text:
            continue

        h = text_hash(text)

        if h not in seen:
            seen.add(h)
            unique.append(d)

    return unique


# =====================================
# cleaning
# =====================================

CLEAN_DOCS = []

for doc in DOCS_NO_HEADERS:   # ← مهم
    cleaned = clean_text(doc["text"])

    if cleaned:
        CLEAN_DOCS.append({
            **doc,
            "text": cleaned
        })

CLEAN_DOCS = dedupe_docs(CLEAN_DOCS)

print(f"Cleaned & deduped -> {len(CLEAN_DOCS)} document(s).")

Cleaned & deduped -> 174 document(s).


In [60]:
# 🔍 Inspection Cell: Compare a page BEFORE vs AFTER cleaning
PAGE_NUMBER_TO_INSPECT = 25  # يمكنك تغيير رقم الصفحة لأي رقم تحبين فحصه (من 1 إلى 174)

page_idx = min(max(0, PAGE_NUMBER_TO_INSPECT - 1), len(RAW_DOCS) - 1)

raw_doc = RAW_DOCS[page_idx]
cleaned_doc = CLEAN_DOCS[page_idx]

print("=" * 70)
print(f"📄 INSPECTING: {raw_doc['source']}")
print("=" * 70)
print(f"📊 Statistics:")
print(f"   - Raw text length     : {len(raw_doc['text'])} characters")
print(f"   - Cleaned text length : {len(cleaned_doc['text'])} characters")
print(f"   - Noise removed       : {len(raw_doc['text']) - len(cleaned_doc['text'])} characters/spaces")
print(f"   - Total duplicates    : {len(RAW_DOCS) - len(CLEAN_DOCS)} duplicate page(s) removed overall")

print("\n" + "-" * 25 + " [1] BEFORE CLEANING (RAW TEXT) " + "-" * 25)
print("--- First 300 characters ---")
print(raw_doc['text'][:500])
print("\n--- Last 200 characters (where footers/page numbers live) ---")
print(raw_doc['text'][-599:])

print("\n" + "-" * 25 + " [2] AFTER CLEANING (CLEANED TEXT) " + "-" * 25)
print("--- First 300 characters ---")
print(cleaned_doc['text'][:500])
print("\n--- Last 200 characters (cleaned footers) ---")
print(cleaned_doc['text'][-500:])
print("=" * 70)

📄 INSPECTING: Handbook on Nuclear Law.pdf (page 25)
📊 Statistics:
   - Raw text length     : 2471 characters
   - Cleaned text length : 2437 characters
   - Noise removed       : 34 characters/spaces
   - Total duplicates    : 0 duplicate page(s) removed overall

------------------------- [1] BEFORE CLEANING (RAW TEXT) -------------------------
--- First 300 characters ---
permission, including ‘authorization’, ‘licence’, ‘permit’, ‘certificate’ or
‘approval’. In applying the permission principle, it is important for the law to
identify clearly those activities or facilities that require an authorization, and
those that do not. In cases in which the regulatory body has found that the risks
associated with an activity are so low as to be below regulatory concern, a
specific authorization may not be required. In such cases a general
authorization can be issued in th

--- Last 200 characters (where footers/page numbers live) ---
lude the potential for such damage,
nuclear law requires tha

## Step 3 — Split text into chunks

Sentence-aware chunking with configurable size and overlap so context isn't cut mid-thought.


In [61]:
import re


def split_sentences(text):
    return [
        s.strip()
        for s in re.split(r"(?<=[.!?])\s+", text)
        if s.strip()
    ]


def split_paragraphs(text):
    lines = text.splitlines()

    paragraphs = []
    current = []

    for line in lines:
        line = line.strip()

        if not line:
            if current:
                paragraphs.append(" ".join(current))
                current = []
            continue

        # New paragraph indicators
        new_paragraph = False

        # Numbered legal paragraphs:
        # 1.1.1. Something
        # 1.1.2. Something
        if re.match(r"^\d+(?:\.\d+)+\.?\s+", line):
            new_paragraph = True

        # Lettered points:
        # (a) Something
        # (b) Something
        if re.match(r"^\([a-zA-Z]\)\s+", line):
            new_paragraph = True

        # Roman/numbered points:
        # (i) Something
        # (ii) Something
        if re.match(r"^\([ivxlcdm]+\)\s+", line, re.I):
            new_paragraph = True

        if new_paragraph and current:
            paragraphs.append(" ".join(current))
            current = []

        current.append(line)

    if current:
        paragraphs.append(" ".join(current))

    return paragraphs


def chunk_text(
    text,
    source,
    max_chars=1000,
    overlap_sentences=1
):
    paragraphs = split_paragraphs(text)
    chunks = []

    for paragraph in paragraphs:

        # Paragraph fits → keep it as one chunk
        if len(paragraph) <= max_chars:
            chunks.append(paragraph)
            continue

        # Paragraph is too large → split into sentences
        sentences = split_sentences(paragraph)
        current = []

        for sent in sentences:

            current_len = sum(len(s) + 1 for s in current)

            if current and current_len + len(sent) > max_chars:
                chunks.append(" ".join(current))

                # Keep last sentence as overlap
                current = current[-overlap_sentences:]

            # Sentence itself is larger than max_chars
            if len(sent) > max_chars:

                if current:
                    chunks.append(" ".join(current))
                    current = []

                for i in range(0, len(sent), max_chars):
                    part = sent[i:i + max_chars].strip()

                    if part:
                        chunks.append(part)

            else:
                current.append(sent)

        if current:
            chunks.append(" ".join(current))

    return [
        {
            "text": chunk,
            "source": source,
            "chunk_id": f"{source}::{i}"
        }
        for i, chunk in enumerate(chunks)
    ]


MAX_CHARS = 1000
OVERLAP_SENTENCES = 1

ALL_CHUNKS = []

for doc in CLEAN_DOCS:
    ALL_CHUNKS.extend(
        chunk_text(
            doc["text"],
            doc["source"],
            MAX_CHARS,
            OVERLAP_SENTENCES
        )
    )

print(
    f"Created {len(ALL_CHUNKS)} chunk(s) "
    f"from {len(CLEAN_DOCS)} document(s)."
)

print("\nExample chunk:\n")
print(ALL_CHUNKS[0]["text"])

Created 772 chunk(s) from 174 document(s).

Example chunk:

Carlton Stoiber Alec Baer Norbert Pelzer Wolfram Tonhauser IAEA International Atomic Energy Agency Handbook on Nuclear Law


In [78]:
# 🔍 Inspection Cell: Display any chunk by its index
CHUNK_INDEX_TO_VIEW = 233 # 👈 غيري هذا الرقم لأي رقم تحبين رؤيته (مثلاً 0, 5, 25, 100...)

if not ALL_CHUNKS:
    print("⚠️ ALL_CHUNKS is empty! Make sure the chunking cell ran successfully.")
elif not (0 <= CHUNK_INDEX_TO_VIEW < len(ALL_CHUNKS)):
    print(f"⚠️ Invalid index! Please choose a number between 0 and {len(ALL_CHUNKS) - 1}.")
else:
    chunk = ALL_CHUNKS[CHUNK_INDEX_TO_VIEW]
    
    print("=" * 70)
    print(f"📦 CHUNK PREVIEW [Index: {CHUNK_INDEX_TO_VIEW} of {len(ALL_CHUNKS) - 1}]")
    print("=" * 70)
    print(f"🆔 Chunk ID    : {chunk.get('chunk_id')}")
    print(f"📄 Source      : {chunk.get('source')}")
    print(f"🏷️ Metadata    : {chunk.get('metadata')}")
    print(f"📏 Length      : {len(chunk['text'])} characters | {len(chunk['text'].split())} words")
    print("-" * 70)
    print("📝 CHUNK TEXT CONTENT:")
    print(chunk['text'])
    print("=" * 70)
    
    # ميزة إضافية: لو الـ Chunk ليس الأول، نتأكد من التداخل (Overlap) مع الـ Chunk السابق
    if CHUNK_INDEX_TO_VIEW > 0:
        prev_chunk = ALL_CHUNKS[CHUNK_INDEX_TO_VIEW - 1]
        if prev_chunk.get('source') == chunk.get('source'):
            print("🔗 Overlap Verification with Previous Chunk:")
            # أخذ أول جملة من الـ chunk الحالي والتحقق هل كانت في نهاية السابق
            first_sentence = chunk['text'].split(".")[0]
            if first_sentence in prev_chunk['text']:
                print("   ✅ Overlap is working properly! (Context preserved from previous chunk).")
            print("=" * 70)

📦 CHUNK PREVIEW [Index: 233 of 771]
🆔 Chunk ID    : Handbook on Nuclear Law.pdf (page 20)::2
📄 Source      : Handbook on Nuclear Law.pdf (page 20)
🏷️ Metadata    : None
📏 Length      : 581 characters | 85 words
----------------------------------------------------------------------
📝 CHUNK TEXT CONTENT:
This hierarchy consists of several levels. The first, usually referred to as the constitutional level, establishes the basic institutional and legal structure governing all relationships in the State. Immediately below the constitutional level is the statutory level, at which specific laws are enacted by a parliament in order to establish other necessary bodies and to adopt measures relating to the broad range of activities affecting national interests. The third level comprises regulations; that is, detailed and often highly technical rules to control or regulate activities
🔗 Overlap Verification with Previous Chunk:
   ✅ Overlap is working properly! (Context preserved from previous chu

## Step 4 — Convert chunks into embeddings

Uses a local `sentence-transformers` model (no API key, runs on CPU) and L2-normalizes vectors so
cosine similarity = dot product (needed for FAISS's `IndexFlatIP`).


In [79]:
from sentence_transformers import SentenceTransformer
import numpy as np

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)

def embed_texts(texts, batch_size=32):
    vecs = embedder.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    return vecs.astype("float32")

chunk_texts = [c["text"] for c in ALL_CHUNKS]

CHUNK_EMBEDDINGS = embed_texts(chunk_texts)

print("Embeddings shape:", CHUNK_EMBEDDINGS.shape)

d:\rag\nuclear-law-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 25/25 [00:21<00:00,  1.17it/s]

Embeddings shape: (772, 384)


## Step 5 — Store embeddings in a vector database

Using **FAISS** here (fast, local, zero setup). The same interface would work with Chroma, Qdrant,
or Pinecone — only this cell would change.


In [80]:
import faiss

dimension = CHUNK_EMBEDDINGS.shape[1]

index = faiss.IndexFlatIP(dimension)  # inner product == cosine similarity on normalized vectors

index.add(CHUNK_EMBEDDINGS)

print(f"FAISS index built with {index.ntotal} vectors of dimension {dimension}.")

# Optional: persist to disk so you don't have to re-embed every run
# faiss.write_index(index, "rag_index.faiss")
# index = faiss.read_index("rag_index.faiss")

# --- Optional: BM25 for hybrid (keyword) search alongside embeddings ---

from rank_bm25 import BM25Okapi

tokenized_corpus = [c["text"].lower().split() for c in ALL_CHUNKS]

bm25 = BM25Okapi(tokenized_corpus)

print("BM25 index built for hybrid search.")

FAISS index built with 772 vectors of dimension 384.
BM25 index built for hybrid search.


## Step 6 — Receive the user's question (optional query rewriting)

For short or ambiguous questions, ask the LLM to rewrite/expand the query into a clearer standalone
search query before retrieval. This is optional — set `REWRITE_QUERY = False` to skip it.


In [81]:
REWRITE_QUERY = True

def rewrite_query(question):
    system = (
        "You rewrite user questions into clear, standalone search queries for a document "
        "retrieval system. Keep it short (1 sentence), preserve all key entities and intent, "
        "and do not answer the question yourself. Return only the rewritten query, nothing else."
    )
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": question},
    ]
    try:
        return chat(messages, temperature=0.0, max_tokens=80).strip()
    except Exception:
        return question  # fall back to the original question if the LLM call fails

def prepare_query(question):
    return rewrite_query(question) if REWRITE_QUERY else question


## Step 7 — Convert the question into an embedding

Uses the **same** embedding model as the chunks — this is required for the similarity search to be
meaningful.


In [82]:
def embed_query(query):
    vec = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    return vec.astype("float32")


## Step 8 — Retrieve the most relevant chunks

Top-K vector search with a similarity threshold, plus an optional hybrid mode that blends in BM25
keyword scores (useful when queries contain exact terms/names/codes the embedding model might blur).


In [83]:
def vector_search(query_vec, top_k=10):
    scores, idxs = index.search(query_vec, top_k)
    results = []
    for score, idx in zip(scores[0], idxs[0]):
        if idx == -1:
            continue
        results.append({**ALL_CHUNKS[idx], "score": float(score)})
    return results

def bm25_search(query, top_k=10):
    scores = bm25.get_scores(query.lower().split())
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [{**ALL_CHUNKS[i], "score": float(scores[i])}
            for i in top_idx if scores[i] > 0]

def hybrid_search(query, query_vec, top_k=10, alpha=0.6):
    """alpha weights vector score vs bm25 score."""
    vec_results = {r["chunk_id"]: r for r in vector_search(query_vec, top_k=top_k * 2)}
    bm25_results = {r["chunk_id"]: r for r in bm25_search(query, top_k=top_k * 2)}

    def normalize(results):
        if not results:
            return {}
        vals = [r["score"] for r in results.values()]
        lo, hi = min(vals), max(vals)
        rng = (hi - lo) or 1.0
        return {k: (r["score"] - lo) / rng for k, r in results.items()}

    vec_norm = normalize(vec_results)
    bm25_norm = normalize(bm25_results)

    all_ids = set(vec_norm) | set(bm25_norm)
    combined = []

    for cid in all_ids:
        score = alpha * vec_norm.get(cid, 0.0) + (1 - alpha) * bm25_norm.get(cid, 0.0)
        chunk = vec_results.get(cid) or bm25_results.get(cid)
        combined.append({**chunk, "score": score})

    combined.sort(key=lambda r: r["score"], reverse=True)
    return combined[:top_k]


TOP_K = 8
USE_HYBRID_SEARCH = True

def retrieve(query, query_vec, top_k=TOP_K):
    if USE_HYBRID_SEARCH:
        return hybrid_search(query, query_vec, top_k=top_k)
    return vector_search(query_vec, top_k=top_k)

## Step 9 — Rerank the retrieved chunks (Cross-Encoder)

A cross-encoder scores `(query, chunk)` pairs jointly, which is much more accurate at judging true
relevance than the bi-encoder similarity score used for the initial retrieval. We keep only the
best `RERANK_TOP_N` chunks after this step.


In [84]:
from sentence_transformers import CrossEncoder

RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
reranker = CrossEncoder(RERANKER_MODEL_NAME)

RERANK_TOP_N = 4

def rerank(query, candidates, top_n=RERANK_TOP_N):
    if not candidates:
        return []

    pairs = [[query, c["text"]] for c in candidates]
    scores = reranker.predict(pairs)

    for c, s in zip(candidates, scores):
        c["rerank_score"] = float(s)

    candidates.sort(key=lambda c: c["rerank_score"], reverse=True)
    return candidates[:top_n]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1553.18it/s]


## Step 10 — Build the LLM context

Merge the question with the reranked chunks, remove near-duplicate content, and format each chunk
with a source tag so the model (and we) can cite it later.


In [85]:
def deduplicate_chunks(chunks, similarity_cutoff=0.92):
    """Remove near-duplicate chunks using cosine similarity."""

    if len(chunks) <= 1:
        return chunks

    texts = [c["text"] for c in chunks]
    vecs = embed_texts(texts)

    keep = []
    kept_vecs = []

    for i, v in enumerate(vecs):
        is_dup = any(
            float(np.dot(v, kv)) > similarity_cutoff
            for kv in kept_vecs
        )

        if not is_dup:
            keep.append(chunks[i])
            kept_vecs.append(v)

    return keep


def build_context(chunks):
    chunks = deduplicate_chunks(chunks)

    numbered = []

    for i, c in enumerate(chunks, start=1):
        numbered.append(
            f"[{i}] (source: {c['source']})\n{c['text']}"
        )

    context_str = "\n\n".join(numbered)

    return context_str, chunks

## Step 11 & 12 — Send context to the LLM, generate the final answer

Strict system prompt: answer **only** from the provided context, cite sources by their `[n]` tag,
and explicitly say when the answer isn't in the context (no guessing / hallucinating).


In [86]:
SYSTEM_PROMPT = """You are a specialized research assistant focused on nuclear law,
nuclear regulation, radiation protection, nuclear safety, nuclear liability,
safeguards, and related legal and regulatory topics.

Answer the user's question using ONLY the information contained in the provided
context chunks.

Follow these rules strictly:

1. Use only the provided context. Do not use outside knowledge or invent information.
2. Answer only questions that are relevant to the subject matter of the provided documents.
3. If the question is unrelated to the documents or the context does not contain
   enough information, say:
   "The provided documents do not contain this information."
4. Cite factual claims using the bracket number of the supporting chunk, e.g. [1].
5. If multiple chunks support a claim, cite all relevant chunks, e.g. [1][3].
6. Do not add a citation unless the cited chunk actually supports the claim.
7. Answer the question directly and concisely, then provide relevant supporting detail.
8. Do not answer general knowledge, casual, personal, or unrelated questions
   using assumptions or outside knowledge.
9. When the documents provide conflicting information, clearly state the conflict
   and cite the relevant sources.
"""

def generate_answer(question, context_str):
    user_prompt = f"""Context:
{context_str}

Question:
{question}

Answer the question according to the system rules."""

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]

    return chat(messages, temperature=0.2, max_tokens=800)

## Full pipeline — put it all together

In [87]:
def rag_answer(question, verbose=True):

    # Step 6: receive + optionally rewrite the question
    search_query = prepare_query(question)

    if verbose and search_query != question:
        print(f"Rewritten query: {search_query}\n")

    # Step 7: embed the question
    q_vec = embed_query(search_query)

    # Step 8: retrieve
    candidates = retrieve(search_query, q_vec)

    if verbose:
        print(f"Retrieved {len(candidates)} candidate chunk(s).")

    # Step 9: rerank
    top_chunks = rerank(search_query, candidates)

    if verbose:
        print(f"Kept top {len(top_chunks)} chunk(s) after reranking.\n")
        for c in top_chunks:
            print(f"  - {c['source']} | rerank_score={c.get('rerank_score', 0):.3f}")

    # Step 9.5: relevance check
    if not top_chunks:
        return {
            "question": question,
            "answer": "The provided documents do not contain this information.",
            "sources": [],
            "chunks_used": [],
        }

    # Step 10: build context
    context_str, final_chunks = build_context(top_chunks)

    # Step 11 & 12: generate the final, cited answer
    answer = generate_answer(question, context_str)

    sources = {c["source"] for c in final_chunks}

    return {
        "question": question,
        "answer": answer,
        "sources": sorted(sources),
        "chunks_used": final_chunks,
    }

## Try it out

Ask a question about whatever documents you loaded in Step 1 (the demo corpus covers RAG, FAISS,
and rerankers if you haven't loaded your own files yet).

In [91]:
result = rag_answer("what is nuclear law?")

print("QUESTION:", result["question"])
print("\nANSWER:\n", result["answer"])
print("\nSOURCES:", result["sources"])


Rewritten query: What is the definition and scope of nuclear law?

Retrieved 8 candidate chunk(s).
Kept top 4 chunk(s) after reranking.

  - Handbook on Nuclear Law.pdf (page 21) | rerank_score=7.086
  - Handbook on Nuclear Law.pdf (page 22) | rerank_score=4.042
  - Handbook on Nuclear Law.pdf (page 20) | rerank_score=3.516
  - Handbook on Nuclear Law.pdf (page 11) | rerank_score=3.078


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.62it/s]


QUESTION: what is nuclear law?

ANSWER:
 According to the provided context, nuclear law can be defined as [1]: The body of special legal norms created to regulate the conduct of legal or natural persons engaged in activities related to fissionable materials, ionizing radiation and exposure to natural sources of radiation. This definition comprises four key elements.

SOURCES: ['Handbook on Nuclear Law.pdf (page 11)', 'Handbook on Nuclear Law.pdf (page 20)', 'Handbook on Nuclear Law.pdf (page 21)', 'Handbook on Nuclear Law.pdf (page 22)']


## Notes & how to extend

- **Switch providers**: edit `LLM_PROVIDER` / `LLM_MODEL` in the config cell — everything else is unchanged.
- **Swap the vector DB**: replace the FAISS cell with a Chroma/Qdrant/Pinecone client that exposes an `add()` and `search()` you call the same way.
- **Better embeddings**: swap `EMBEDDING_MODEL_NAME` for a larger model (e.g. `BAAI/bge-large-en-v1.5`) if accuracy matters more than speed.
- **Persist the index**: uncomment the `faiss.write_index` / `read_index` lines so you don't re-embed on every run.
- **Streaming answers**: add `"stream": True` to the `chat()` payload and iterate over `resp.iter_lines()` if you want token-by-token output in a UI.


## Step 13 — Deployment Preparation (Cloudflare Web App)

This step bridges the Jupyter Notebook with the web app deployment according to **`Deploying_RAG_Notebook_Free_Web_App.pdf`**:
1. **`export_for_deployment()`** saves the processed knowledge base (`knowledge_base.json`) for the Cloudflare Worker.
2. **`query_rag(question)`** provides the exact clean standalone signature required by Section 1 of the guide.


In [ ]:
import json
from pathlib import Path

# 1. Export knowledge base for the Cloudflare Worker backend
def export_for_deployment(output_path="knowledge_base.json"):
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(ALL_CHUNKS, f, ensure_ascii=False, indent=2)
    print(f"Exported {len(ALL_CHUNKS)} chunks to {output_path} successfully!")

export_for_deployment("knowledge_base.json")
export_for_deployment("worker/src/knowledge_base.json")

# 2. Standalone public function as specified in Step 1 of the deployment guide
def query_rag(question: str) -> dict:
    res = rag_answer(question, verbose=False)
    return {
        "answer": res["answer"],
        "sources": res["sources"]
    }

# Quick test of query_rag()
test_output = query_rag("What is the definition of nuclear law?")
print("\n--- [query_rag output] ---")
print("Answer:", test_output["answer"][:200], "...")
print("Sources:", test_output["sources"])


## Step 14 — Nuclear Domain Detection & Multi-User Workspaces

This section tests and demonstrates the dynamic multi-document and multi-user isolation features:
1. **`verify_nuclear_domain()`**: Verifies that any uploaded PDF genuinely belongs to Nuclear Law and Radiation Safety before accepting it.
2. **`workspace_manager`**: Keeps documents and search indexes partitioned by user / workspace ID.


In [ ]:
from domain_detector import verify_nuclear_domain
from workspace_manager import process_and_add_pdf, get_workspace_catalog, search_workspace_chunks

# 1. Test Domain Classifier on valid Nuclear Law text vs non-nuclear text
valid_text = "Article 5: The national regulatory body shall establish regulations for radiation protection, licensing of nuclear installations, and safety standards for radioactive waste."
invalid_text = "Learn digital photography: adjust shutter speed, ISO, and aperture to capture portrait lighting in outdoor photography."

print("--- [Domain Test 1: Nuclear Law Text] ---")
print(verify_nuclear_domain(valid_text))

print("\n--- [Domain Test 2: Photography Text (Should Reject)] ---")
print(verify_nuclear_domain(invalid_text))

# 2. Demonstrate Multi-User Workspace Isolation
catalog_user_a = get_workspace_catalog("user_alice")
catalog_user_b = get_workspace_catalog("user_bob")
print(f"\nUser Alice catalog count: {len(catalog_user_a)}")
print(f"User Bob catalog count  : {len(catalog_user_b)}")
